# Benchmark de Quantum Simulator en AMD ROCm

Este notebook ejecuta el benchmark del simulador statevector en la GPU de AMD (W7900D, RDNA 3, gfx1100) detectando el hardware en runtime y generando archivos de salida para registrar los resultados.

**Flujo:**
1. Clona el repositorio del hackathon desde GitHub.
2. Instala dependencias.
3. Detecta hardware (GPU, arquitectura GFX, versión ROCm).
4. Ejecuta barrido de benchmarks (n × batch × dtype).
5. Genera `benchmark_results_*.json` (datos brutos) y `BENCHMARK_SUMMARY_*.md` (resumen legible).

**IMPORTANTE:** Guarda los archivos generados bajo `/persistent` (o `/workspace` según tu URL de sesión) para que sobrevivan al apagar la sesión.

In [ ]:
# --- Paso 1: Clonar el repositorio del hackathon ---
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/jalp17/chern-calibration-copilot.git"
REPO_DIR = Path("./chern-calibration-copilot")

if not REPO_DIR.exists():
    print(f"Clonando {REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Fallo al clonar el repositorio")
else:
    print(f"El repositorio ya existe en {REPO_DIR}")
    result = subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull"],
        capture_output=True, text=True
    )
    print(result.stdout)

os.chdir(str(REPO_DIR / "entrega"))
print(f"Directorio de trabajo: {os.getcwd()}")
print("Archivos:", os.listdir(".")[:20])

In [ ]:
# --- Paso 2: Instalar dependencias ---
import subprocess, sys

print("Instalando dependencias...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Fallo al instalar dependencias")
print("✓ Dependencias instaladas")

In [ ]:
# --- Paso 3: Detectar hardware en runtime ---
import sys
sys.path.insert(0, ".")
sys.path.insert(0, "./simulador")

from hardware_wheels import detect_gpu_architecture, install_command, quick_summary, print_report

print("=" * 70)
print("DETECCIÓN DE HARDWARE (runtime - no compila nada)")
print("=" * 70)

gpu = detect_gpu_architecture()
hw_summary = quick_summary()

print(f"GPU             : {gpu.name}")
print(f"Arquitectura    : {gpu.architecture}")
print(f"Versión ROCm    : {gpu.rocm_version}")
print(f"Es MI300 (gfx942): {gpu.is_mi300}")
print(f"Es gfx1100      : {gpu.is_gfx1100}")
print(f"Wheel precompilada: {'SÍ' if gpu.supports_precompiled else 'NO'}")
print()
print("Comando de instalación de pennylane-lightning-gpu (bajo demanda):")
print(f"  {install_command(gpu)}")
print()
print("Nota: el simulador PyTorch puro (qsim_torch.py) funciona sin pennylane-lightning-gpu.")
print("Solo instálalo si quieres comparar con lightning.amdgpu.")
print("=" * 70)

In [ ]:
# --- Paso 4: Ejecutar benchmark ---
from benchmark_notebook import run_benchmark

# Configuración para GPU (W7900D o lo que detecte)
resultados = run_benchmark(
    device_str='cuda',        # 'cuda' para GPU AMD/ROCm, 'cpu' para CPU
    ns=[16, 20, 24, 26, 28],  # qubits a probar
    batches=[1, 256, 1024],   # tamaños de lote
    dtypes=None,              # [complex128, complex64] por defecto
    output_dir='.',           # guardar en el directorio actual
)

In [ ]:
# --- Paso 5: Mostrar archivos generados y resumen ---
from pathlib import Path

out = Path(resultados['output_files']['json'])
md = Path(resultados['output_files']['markdown'])

print("=" * 70)
print("ARCHIVOS GENERADOS")
print("=" * 70)
print(f"JSON (datos brutos):   {out}  ({out.stat().st_size / 1024:.1f} KB)")
print(f"Markdown (resumen):    {md}  ({md.stat().st_size / 1024:.1f} KB)")
print()
print("Contenido del resumen:")
print("=" * 70)
print(md.read_text(encoding='utf-8'))

print("=" * 70)
print("INSTRUCCIONES PARA DESCARGAR")
print("=" * 70)
print("1. Busca los archivos benchmark_results_*.json y BENCHMARK_SUMMARY_*.md")
print("   en el explorador de archivos de Jupyter.")
print("2. Haz clic derecho → Download para bajarlos a tu máquina local.")
print("3. El JSON contiene todos los datos brutos. El Markdown tiene el resumen.")
print()
print("⚠️ IMPORTANTE: Apaga la sesión con 'Turn-off Session' cuando termines.")
print("   La cuota de GPU es 3 h/día y corre aunque la sesión esté inactiva.")

In [ ]:
# Celda de diagnóstico rápido (opcional)
import torch
if torch.cuda.is_available():
    print(f"GPU activa: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
    print(f"ROCm/HIP: {torch.version.hip}")
else:
    print("No hay GPU disponible. Ejecutando en CPU.")
